In [7]:

import os
import sys
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# ==========================================
# PATHS
# ==========================================
DATA_DIR = "/kaggle/input/datasets/tvjpro2/inter-iit-cv-ps2"
OUT_DIR = "/kaggle/working/task5"

os.makedirs(OUT_DIR, exist_ok=True)

# ==========================================
# REPRODUCIBILITY
# ==========================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Dataset files:", os.listdir(DATA_DIR))

# ==========================================
# LOAD DATA
# ==========================================
train_images = np.load(os.path.join(DATA_DIR, "train_images.npy"))
test_images = np.load(os.path.join(DATA_DIR, "test_images.npy"))

labels_df = pd.read_csv(os.path.join(DATA_DIR, "labels.csv"))

train_idx = np.loadtxt(
    os.path.join(DATA_DIR, "train_idx.txt"), dtype=int
)
val_idx = np.loadtxt(
    os.path.join(DATA_DIR, "val_idx.txt"), dtype=int
)

# IMPORTANT: Task 5 uses synthetic_label, not noisy_label.
labels = labels_df.set_index("index")["synthetic_label"]

train_labels = labels.loc[train_idx].to_numpy(dtype=np.int64)
val_labels = labels.loc[val_idx].to_numpy(dtype=np.int64)

print("Train images:", len(train_idx))
print("Validation images:", len(val_idx))
print("Test images:", len(test_images))
print("Synthetic label range:",
      train_labels.min(), train_labels.max())

# ==========================================
# TRANSFORMS
# ==========================================
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor()
])

# ==========================================
# DATASET
# ==========================================
class CIFARArrayDataset(Dataset):
    def __init__(self, images, indices, labels=None, transform=None):
        self.images = images
        self.indices = np.asarray(indices, dtype=np.int64)
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        image = self.images[idx]

        if self.transform:
            image = self.transform(image)

        if self.labels is None:
            return image, idx

        return image, int(self.labels[i])

train_dataset = CIFARArrayDataset(
    train_images, train_idx, train_labels, train_transform
)

val_dataset = CIFARArrayDataset(
    train_images, val_idx, val_labels, eval_transform
)

train_loader = DataLoader(
    train_dataset, batch_size=128, shuffle=True,
    num_workers=2, pin_memory=True
)

val_loader = DataLoader(
    val_dataset, batch_size=256, shuffle=False,
    num_workers=2, pin_memory=True
)

# ==========================================
# PROVIDED RESNET-18 — TRAIN FROM SCRATCH
# ==========================================
sys.path.insert(0, DATA_DIR)
from model import ResNet18

model = ResNet18(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4
)

EPOCHS = 60
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

# ==========================================
# TRAINING
# ==========================================
best_val_acc = -1.0
history = []

best_path = os.path.join(OUT_DIR, "t5_baseline_synth_best.pth")
latest_path = os.path.join(OUT_DIR, "t5_baseline_synth_latest.pth")
history_path = os.path.join(OUT_DIR, "t5_baseline_synth_history.csv")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, targets in train_loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * targets.size(0)
        train_correct += (logits.argmax(1) == targets).sum().item()
        train_total += targets.size(0)

    scheduler.step()

    model.eval()
    val_correct = 0
    val_total = 0
    val_loss_sum = 0.0

    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, targets)

            val_loss_sum += loss.item() * targets.size(0)
            val_correct += (logits.argmax(1) == targets).sum().item()
            val_total += targets.size(0)

    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    train_loss = train_loss_sum / train_total
    val_loss = val_loss_sum / val_total

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc_synthetic": val_acc
    }
    history.append(row)

    # Save latest checkpoint every epoch for recovery
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_acc": max(best_val_acc, val_acc)
    }, latest_path)

    pd.DataFrame(history).to_csv(history_path, index=False)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_path)
        print(f"  Saved best checkpoint: {best_path}")

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train loss {train_loss:.4f} | "
        f"Train acc {train_acc:.4%} | "
        f"Synthetic val loss {val_loss:.4f} | "
        f"Synthetic val acc {val_acc:.4%} | "
        f"Best {best_val_acc:.4%}"
    )

print("\nBaseline training complete.")
print("Best synthetic validation accuracy:", f"{best_val_acc:.4%}")
print("Best checkpoint:", best_path)
print("History:", history_path)


Device: cuda
Dataset files: ['check_format.py', 'classes.txt', 'labels.csv', 'model.py', 'results_template.json', 't1_clean_ref.csv', 't1_clean_ref_model.pth', 't1_noisy_ce.csv', 't1_noisy_ce_model.pth', 't2_dynamics.csv', 't2_early_stop.csv', 't3_noise_scores_improved.csv', 'test_images.npy', 'train_idx.txt', 'train_images.npy', 'val_idx.txt']
Train images: 45000
Validation images: 5000
Test images: 10000
Synthetic label range: 0 9
  Saved best checkpoint: /kaggle/working/task5/t5_baseline_synth_best.pth
Epoch 01/60 | Train loss 2.3287 | Train acc 17.2044% | Synthetic val loss 2.1692 | Synthetic val acc 22.2400% | Best 22.2400%
  Saved best checkpoint: /kaggle/working/task5/t5_baseline_synth_best.pth
Epoch 02/60 | Train loss 2.1623 | Train acc 22.6756% | Synthetic val loss 2.1527 | Synthetic val acc 23.6000% | Best 23.6000%
  Saved best checkpoint: /kaggle/working/task5/t5_baseline_synth_best.pth
Epoch 03/60 | Train loss 2.1259 | Train acc 25.3511% | Synthetic val loss 2.0945 | Synthe

In [9]:

import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

DATA_DIR = "/kaggle/input/datasets/tvjpro2/inter-iit-cv-ps2"
OUT_DIR = "/kaggle/working/task5"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_images = np.load(os.path.join(DATA_DIR, "test_images.npy"))

eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor()
])

class TestDataset(Dataset):
    def __init__(self, images, transform):
        self.images = images
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.transform(self.images[idx]), idx

test_loader = DataLoader(
    TestDataset(test_images, eval_transform),
    batch_size=256,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

from model import ResNet18

model = ResNet18(num_classes=10).to(device)

checkpoint_path = os.path.join(
    OUT_DIR, "t5_baseline_synth_best.pth"
)
model.load_state_dict(
    torch.load(checkpoint_path, map_location=device, weights_only=True)
)
model.eval()

all_ids = []
all_preds = []

with torch.no_grad():
    for images, ids in test_loader:
        images = images.to(device, non_blocking=True)
        predictions = model(images).argmax(dim=1).cpu().numpy()

        all_ids.extend(ids.numpy().tolist())
        all_preds.extend(predictions.tolist())

submission = pd.DataFrame({
    "id": all_ids,
    "pred": all_preds
})

output_path = os.path.join(OUT_DIR, "t5_baseline_synth.csv")
submission.to_csv(output_path, index=False)

# Validate submission format
assert len(submission) == 10000, "Expected 10,000 predictions"
assert submission["id"].is_unique, "Duplicate test IDs found"
assert submission["id"].min() == 0 and submission["id"].max() == 9999
assert submission["pred"].between(0, 9).all()

print(submission.head())
print("Shape:", submission.shape)
print("Saved to:", output_path)
print("Format validation passed.")


   id  pred
0   0     0
1   1     8
2   2     8
3   3     3
4   4     3
Shape: (10000, 2)
Saved to: /kaggle/working/task5/t5_baseline_synth.csv
Format validation passed.


In [2]:

import os
import sys
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ==================================================
# 1. AUTOMATICALLY DETECT DATASET DIRECTORY
# ==================================================
DATA_DIR = None
required_files = {
    "train_images.npy",
    "test_images.npy",
    "labels.csv",
    "model.py",
    "train_idx.txt",
    "val_idx.txt"
}

for root, dirs, files in os.walk("/kaggle/input"):
    if required_files.issubset(set(files)):
        DATA_DIR = root
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Dataset not found. Add the Inter IIT CV PS2 dataset "
        "using Kaggle's Add Input option."
    )

print("Dataset directory:", DATA_DIR)

OUT_DIR = "/kaggle/working/task5/label_smoothing"
os.makedirs(OUT_DIR, exist_ok=True)

# ==================================================
# 2. REPRODUCIBILITY AND DEVICE
# ==================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ==================================================
# 3. LOAD DATA AND SYNTHETIC LABELS
# ==================================================
train_images = np.load(os.path.join(DATA_DIR, "train_images.npy"))
labels_df = pd.read_csv(os.path.join(DATA_DIR, "labels.csv"))

train_idx = np.loadtxt(
    os.path.join(DATA_DIR, "train_idx.txt"), dtype=int
)
val_idx = np.loadtxt(
    os.path.join(DATA_DIR, "val_idx.txt"), dtype=int
)

label_map = labels_df.set_index("index")["synthetic_label"]

train_labels = label_map.loc[train_idx].to_numpy(dtype=np.int64)
val_labels = label_map.loc[val_idx].to_numpy(dtype=np.int64)

assert len(train_idx) == len(train_labels)
assert len(val_idx) == len(val_labels)
assert train_labels.min() >= 0 and train_labels.max() <= 9
assert val_labels.min() >= 0 and val_labels.max() <= 9

print("Training examples:", len(train_idx))
print("Validation examples:", len(val_idx))
print("Training labels: synthetic_label")
print("Validation labels: synthetic_label")

# ==================================================
# 4. TRANSFORMS
# ==================================================
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor()
])

# ==================================================
# 5. DATASET
# ==================================================
class CIFARArrayDataset(Dataset):
    def __init__(self, images, indices, labels, transform):
        self.images = images
        self.indices = np.asarray(indices, dtype=np.int64)
        self.labels = np.asarray(labels, dtype=np.int64)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        image = self.transform(self.images[idx])
        label = int(self.labels[i])
        return image, label

train_dataset = CIFARArrayDataset(
    train_images, train_idx, train_labels, train_transform
)
val_dataset = CIFARArrayDataset(
    train_images, val_idx, val_labels, eval_transform
)

train_loader = DataLoader(
    train_dataset, batch_size=128, shuffle=True,
    num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=256, shuffle=False,
    num_workers=2, pin_memory=True
)

# ==================================================
# 6. PROVIDED RESNET-18, TRAINED FROM SCRATCH
# ==================================================
sys.path.insert(0, DATA_DIR)
from model import ResNet18

model = ResNet18(num_classes=10).to(device)

# Label smoothing method
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=5e-4
)

EPOCHS = 60
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

# ==================================================
# 7. OUTPUT FILES
# ==================================================
best_path = os.path.join(
    OUT_DIR, "t5_label_smoothing_best.pth"
)
latest_path = os.path.join(
    OUT_DIR, "t5_label_smoothing_latest.pth"
)
history_path = os.path.join(
    OUT_DIR, "t5_label_smoothing_history.csv"
)

best_val_acc = -1.0
history = []

# ==================================================
# 8. TRAINING LOOP
# ==================================================
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, targets in train_loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * targets.size(0)
        train_correct += (
            logits.argmax(dim=1) == targets
        ).sum().item()
        train_total += targets.size(0)

    scheduler.step()

    # Validation: synthetic labels only
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, targets)

            val_loss_sum += loss.item() * targets.size(0)
            val_correct += (
                logits.argmax(dim=1) == targets
            ).sum().item()
            val_total += targets.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total
    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc_synthetic": train_acc,
        "val_loss": val_loss,
        "val_acc_synthetic": val_acc
    })

    # Save latest checkpoint for recovery
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_acc": max(best_val_acc, val_acc)
    }, latest_path)

    pd.DataFrame(history).to_csv(history_path, index=False)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_acc_synthetic": val_acc
        }, best_path)

        print("  Best checkpoint saved.")

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train loss={train_loss:.4f} | "
        f"Train acc={train_acc:.2%} | "
        f"Synthetic val acc={val_acc:.2%} | "
        f"Best={best_val_acc:.2%}"
    )

print("\nLabel smoothing training completed.")
print(f"Best synthetic validation accuracy: {best_val_acc:.2%}")
print("Best checkpoint:", best_path)
print("History:", history_path)


Dataset directory: /kaggle/input/datasets/tvjpro2/inter-iit-cv-ps2
Device: cuda
Training examples: 45000
Validation examples: 5000
Training labels: synthetic_label
Validation labels: synthetic_label
  Best checkpoint saved.
Epoch 01/60 | Train loss=2.3377 | Train acc=17.65% | Synthetic val acc=22.70% | Best=22.70%
Epoch 02/60 | Train loss=2.1886 | Train acc=22.82% | Synthetic val acc=20.70% | Best=22.70%
  Best checkpoint saved.
Epoch 03/60 | Train loss=2.1579 | Train acc=25.32% | Synthetic val acc=25.86% | Best=25.86%
Epoch 04/60 | Train loss=2.1264 | Train acc=28.00% | Synthetic val acc=25.06% | Best=25.86%
Epoch 05/60 | Train loss=2.0974 | Train acc=30.21% | Synthetic val acc=24.86% | Best=25.86%
  Best checkpoint saved.
Epoch 06/60 | Train loss=2.0732 | Train acc=32.39% | Synthetic val acc=28.82% | Best=28.82%
Epoch 07/60 | Train loss=2.0429 | Train acc=34.68% | Synthetic val acc=28.30% | Best=28.82%
  Best checkpoint saved.
Epoch 08/60 | Train loss=2.0137 | Train acc=37.22% | Synt

In [3]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

def find_file(filename):
    for root in (Path("/kaggle/working"), Path("/kaggle/input")):
        matches = list(root.rglob(filename))
        if matches:
            return matches[0]
    raise FileNotFoundError(
        f"Could not find {filename} under /kaggle/working or /kaggle/input"
    )

checkpoint_path = find_file("t5_label_smoothing_best.pth")
model_path = find_file("model.py")

if "test_images" not in globals():
    test_images_path = find_file("test_images.npy")
    test_images = np.load(test_images_path, mmap_mode="r")

sys.path.insert(0, str(model_path.parent))
from model import ResNet18

if test_images.shape != (10_000, 32, 32, 3):
    raise ValueError(f"Unexpected test image shape: {test_images.shape}")

class TestImages(Dataset):
    def __init__(self, images):
        self.images = images

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        image = np.asarray(self.images[i])
        if image.dtype != np.uint8:
            image = np.clip(image, 0, 255).astype(np.uint8)
        return torch.from_numpy(image.copy()).permute(2, 0, 1).float() / 255.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet18(num_classes=10).to(device)

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    checkpoint = checkpoint["model_state_dict"]
elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
    checkpoint = checkpoint["state_dict"]

model.load_state_dict(checkpoint)
model.eval()

loader = DataLoader(TestImages(test_images), batch_size=256, shuffle=False)
predictions = []

with torch.inference_mode():
    for batch in loader:
        predictions.extend(model(batch.to(device)).argmax(dim=1).cpu().tolist())

assert len(predictions) == 10_000
assert all(0 <= pred <= 9 for pred in predictions)

output_path = Path("/kaggle/working/t5_label_smoothing.csv")
pd.DataFrame({
    "id": np.arange(len(predictions)),
    "pred": predictions
}).to_csv(output_path, index=False)

print(f"Checkpoint: {checkpoint_path}")
print(f"Saved: {output_path}")
print(pd.read_csv(output_path).head())

Checkpoint: /kaggle/working/task5/label_smoothing/t5_label_smoothing_best.pth
Saved: /kaggle/working/t5_label_smoothing.csv
   id  pred
0   0     8
1   1     8
2   2     8
3   3     3
4   4     9


In [4]:

import os
import sys
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# ==================================================
# 1. FIND DATASET
# ==================================================
DATA_DIR = None
required = {
    "train_images.npy", "test_images.npy", "labels.csv",
    "model.py", "train_idx.txt", "val_idx.txt"
}

for root, dirs, files in os.walk("/kaggle/input"):
    if required.issubset(set(files)):
        DATA_DIR = root
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Dataset not found. Attach the Inter IIT CV PS2 dataset "
        "through Kaggle's Add Input panel."
    )

print("Dataset:", DATA_DIR)

OUT_DIR = "/kaggle/working/task5/filtering"
os.makedirs(OUT_DIR, exist_ok=True)

# ==================================================
# 2. CONFIGURATION
# ==================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

THRESHOLD = 0.7
EPOCHS = 60
BATCH_SIZE = 128

# ==================================================
# 3. FIND TASK 3 SCORES
# ==================================================
score_candidates = [
    os.path.join(DATA_DIR, "t3_noise_scores_improved.csv"),
    os.path.join(DATA_DIR, "t3_noise_scores.csv"),
]

# Also search attached Kaggle input datasets
for root, dirs, files in os.walk("/kaggle/input"):
    for filename in files:
        if filename in (
            "t3_noise_scores_improved.csv",
            "t3_noise_scores.csv"
        ):
            path = os.path.join(root, filename)
            if path not in score_candidates:
                score_candidates.append(path)

score_path = next(
    (p for p in score_candidates if os.path.isfile(p)), None
)

if score_path is None:
    raise FileNotFoundError(
        "Task 3 score CSV not found. Add your saved "
        "t3_noise_scores_improved.csv as a Kaggle dataset input, "
        "or copy it into the dataset directory."
    )

print("Using Task 3 scores:", score_path)

scores_df = pd.read_csv(score_path)
assert {"index", "score"}.issubset(scores_df.columns)

score_map = scores_df.set_index("index")["score"]

# ==================================================
# 4. LOAD IMAGES, SPLITS AND SYNTHETIC LABELS
# ==================================================
train_images = np.load(os.path.join(DATA_DIR, "train_images.npy"))
labels_df = pd.read_csv(os.path.join(DATA_DIR, "labels.csv"))

train_idx = np.loadtxt(
    os.path.join(DATA_DIR, "train_idx.txt"), dtype=int
)
val_idx = np.loadtxt(
    os.path.join(DATA_DIR, "val_idx.txt"), dtype=int
)

synthetic_map = labels_df.set_index("index")["synthetic_label"]

# Apply the same Task 4 filtering rule: keep score <= 0.7.
missing_scores = set(train_idx.tolist()) - set(score_map.index)
if missing_scores:
    raise ValueError(
        f"Noise scores are missing {len(missing_scores)} training indices."
    )

train_scores = score_map.loc[train_idx].to_numpy(dtype=float)
keep_mask = train_scores <= THRESHOLD
filtered_train_idx = train_idx[keep_mask]

if len(filtered_train_idx) == 0:
    raise ValueError("Filtering removed every training example.")

filtered_train_labels = synthetic_map.loc[
    filtered_train_idx
].to_numpy(dtype=np.int64)

val_labels = synthetic_map.loc[val_idx].to_numpy(dtype=np.int64)

print(f"Original training examples: {len(train_idx)}")
print(f"Retained examples: {len(filtered_train_idx)}")
print(f"Removed examples: {len(train_idx) - len(filtered_train_idx)}")
print(f"Retention rate: {len(filtered_train_idx)/len(train_idx):.2%}")
print("Training labels: synthetic_label")
print("Validation labels: synthetic_label")

# Save the retained indices for reproducibility
pd.DataFrame({"index": filtered_train_idx}).to_csv(
    os.path.join(OUT_DIR, "retained_train_indices.csv"),
    index=False
)

# ==================================================
# 5. TRANSFORMS AND DATASET
# ==================================================
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor()
])

class CIFARArrayDataset(Dataset):
    def __init__(self, images, indices, labels, transform):
        self.images = images
        self.indices = np.asarray(indices, dtype=np.int64)
        self.labels = np.asarray(labels, dtype=np.int64)
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        return self.transform(self.images[idx]), int(self.labels[i])

train_dataset = CIFARArrayDataset(
    train_images, filtered_train_idx,
    filtered_train_labels, train_transform
)

val_dataset = CIFARArrayDataset(
    train_images, val_idx, val_labels, eval_transform
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=256, shuffle=False,
    num_workers=2, pin_memory=True
)

# ==================================================
# 6. PROVIDED RESNET-18 FROM SCRATCH
# ==================================================
sys.path.insert(0, DATA_DIR)
from model import ResNet18

model = ResNet18(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(
    model.parameters(), lr=0.1,
    momentum=0.9, weight_decay=5e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

best_path = os.path.join(OUT_DIR, "t5_filtering_best.pth")
latest_path = os.path.join(OUT_DIR, "t5_filtering_latest.pth")
history_path = os.path.join(OUT_DIR, "t5_filtering_history.csv")

best_val_acc = -1.0
history = []

# ==================================================
# 7. TRAIN AND SELECT USING SYNTHETIC VALIDATION
# ==================================================
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, targets in train_loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * targets.size(0)
        train_correct += (
            logits.argmax(1) == targets
        ).sum().item()
        train_total += targets.size(0)

    scheduler.step()
    model.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, targets)

            val_loss_sum += loss.item() * targets.size(0)
            val_correct += (
                logits.argmax(1) == targets
            ).sum().item()
            val_total += targets.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total
    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc_synthetic": train_acc,
        "val_loss": val_loss,
        "val_acc_synthetic": val_acc
    })

    # Latest checkpoint for recovery
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "best_val_acc": max(best_val_acc, val_acc)
    }, latest_path)

    pd.DataFrame(history).to_csv(history_path, index=False)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_acc_synthetic": val_acc
        }, best_path)
        print("  Best checkpoint saved.")

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train loss={train_loss:.4f} | "
        f"Train acc={train_acc:.2%} | "
        f"Synthetic val acc={val_acc:.2%} | "
        f"Best={best_val_acc:.2%}"
    )

print("\nTask 5 filtering training completed.")
print(f"Best synthetic validation accuracy: {best_val_acc:.2%}")
print("Best checkpoint:", best_path)
print("History:", history_path)


Dataset: /kaggle/input/datasets/tvjpro2/inter-iit-cv-ps2
Device: cuda
Using Task 3 scores: /kaggle/input/datasets/tvjpro2/inter-iit-cv-ps2/t3_noise_scores_improved.csv
Original training examples: 45000
Retained examples: 39118
Removed examples: 5882
Retention rate: 86.93%
Training labels: synthetic_label
Validation labels: synthetic_label
  Best checkpoint saved.
Epoch 01/60 | Train loss=2.3379 | Train acc=16.47% | Synthetic val acc=16.96% | Best=16.96%
  Best checkpoint saved.
Epoch 02/60 | Train loss=2.1626 | Train acc=22.37% | Synthetic val acc=25.10% | Best=25.10%
Epoch 03/60 | Train loss=2.1205 | Train acc=25.28% | Synthetic val acc=23.64% | Best=25.10%
  Best checkpoint saved.
Epoch 04/60 | Train loss=2.0881 | Train acc=27.70% | Synthetic val acc=29.62% | Best=29.62%
Epoch 05/60 | Train loss=2.0566 | Train acc=30.11% | Synthetic val acc=28.76% | Best=29.62%
  Best checkpoint saved.
Epoch 06/60 | Train loss=2.0239 | Train acc=32.40% | Synthetic val acc=31.44% | Best=31.44%
  Best 

In [5]:

import os
import sys
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Reuse DATA_DIR, OUT_DIR and device from the training cell.
test_images = np.load(os.path.join(DATA_DIR, "test_images.npy"))

eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor()
])

class TestDataset(Dataset):
    def __init__(self, images, transform):
        self.images = images
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.transform(self.images[idx]), idx

test_loader = DataLoader(
    TestDataset(test_images, eval_transform),
    batch_size=256, shuffle=False,
    num_workers=2, pin_memory=True
)

sys.path.insert(0, DATA_DIR)
from model import ResNet18

model = ResNet18(num_classes=10).to(device)
checkpoint = torch.load(
    os.path.join(OUT_DIR, "t5_filtering_best.pth"),
    map_location=device,
    weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

ids, predictions = [], []

with torch.no_grad():
    for images, batch_ids in test_loader:
        images = images.to(device, non_blocking=True)
        preds = model(images).argmax(1).cpu().numpy()

        ids.extend(batch_ids.numpy().astype(int).tolist())
        predictions.extend(preds.astype(int).tolist())

submission = pd.DataFrame({"id": ids, "pred": predictions})
submission = submission.sort_values("id").reset_index(drop=True)

assert submission.shape == (10000, 2)
assert submission["id"].is_unique
assert submission["id"].tolist() == list(range(10000))
assert submission["pred"].between(0, 9).all()

output_path = os.path.join(OUT_DIR, "t5_filtering.csv")
submission.to_csv(output_path, index=False)

print(submission.head())
print("Shape:", submission.shape)
print("Saved:", output_path)


   id  pred
0   0     0
1   1     8
2   2     9
3   3     5
4   4     8
Shape: (10000, 2)
Saved: /kaggle/working/task5/filtering/t5_filtering.csv


In [1]:

# TASK 5 — CO-TEACHING ON SYNTHETIC LABELS
# Trains two ResNet-18 networks from scratch.
# Does NOT use clean_label.

import os
import glob
import random
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# ---------------- CONFIG ----------------
SEED = 42
BATCH_SIZE = 128
EPOCHS = 60
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
MAX_FORGET_RATE = 0.20
RAMPUP_EPOCHS = 10
NUM_WORKERS = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Find the dataset folder without assuming its Kaggle slug.
required = [
    "labels.csv", "train_images.npy", "test_images.npy",
    "train_idx.txt", "val_idx.txt", "model.py"
]
candidates = ["/kaggle/working", "/kaggle/input"]

for root in ["/kaggle/input", "/kaggle/working"]:
    for base, dirs, files in os.walk(root):
        if all(x in files for x in required):
            candidates.insert(0, base)
            break

DATA_DIR = next(
    (p for p in dict.fromkeys(candidates)
     if os.path.isdir(p) and all(os.path.exists(os.path.join(p, f))
                                 for f in required)),
    None
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find the folder containing labels.csv, train_images.npy, "
        "test_images.npy, train_idx.txt, val_idx.txt and model.py. "
        "Check that the dataset is attached to the notebook."
    )

print("Dataset:", DATA_DIR)

# Import the provided ResNet18 architecture.
import sys
if DATA_DIR not in sys.path:
    sys.path.insert(0, DATA_DIR)
from model import ResNet18

OUT_DIR = "/kaggle/working/task5_coteaching"
os.makedirs(OUT_DIR, exist_ok=True)

# Optional persistent output: mount Google Drive in a separate cell first
# if you want a second copy saved there.
DRIVE_OUT = "/content/drive/MyDrive/InterIIT_CV/Task5/coteaching"

# ---------------- DATA ----------------
labels = pd.read_csv(os.path.join(DATA_DIR, "labels.csv"))
train_images = np.load(os.path.join(DATA_DIR, "train_images.npy"))
test_images = np.load(os.path.join(DATA_DIR, "test_images.npy"))

train_idx = np.loadtxt(
    os.path.join(DATA_DIR, "train_idx.txt"), dtype=int
).reshape(-1)
val_idx = np.loadtxt(
    os.path.join(DATA_DIR, "val_idx.txt"), dtype=int
).reshape(-1)

# Use only synthetic labels for both training and validation.
synthetic_by_idx = labels.set_index("index")["synthetic_label"]
y_train = synthetic_by_idx.loc[train_idx].to_numpy(dtype=np.int64)
y_val = synthetic_by_idx.loc[val_idx].to_numpy(dtype=np.int64)

assert len(train_idx) == len(y_train)
assert len(val_idx) == len(y_val)
assert set(np.unique(y_train)).issubset(set(range(10)))
assert set(np.unique(y_val)).issubset(set(range(10)))

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])
eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])

class CIFARArrayDataset(Dataset):
    def __init__(self, images, indices, targets=None, transform=None):
        self.images = images
        self.indices = np.asarray(indices)
        self.targets = targets
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        x = self.images[idx]
        if self.transform:
            x = self.transform(x)
        if self.targets is None:
            return x, idx
        return x, int(self.targets[i])

train_ds = CIFARArrayDataset(
    train_images, train_idx, y_train, train_transform
)
val_ds = CIFARArrayDataset(
    train_images, val_idx, y_val, eval_transform
)
test_ds = CIFARArrayDataset(
    test_images, np.arange(len(test_images)), None, eval_transform
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(),
    drop_last=False
)
val_loader = DataLoader(
    val_ds, batch_size=256, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_ds, batch_size=256, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)

# ---------------- MODELS ----------------
net1 = ResNet18(num_classes=10).to(device)
net2 = ResNet18(num_classes=10).to(device)

opt1 = torch.optim.SGD(
    net1.parameters(), lr=LR, momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY
)
opt2 = torch.optim.SGD(
    net2.parameters(), lr=LR, momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY
)
sched1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt1, T_max=EPOCHS
)
sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt2, T_max=EPOCHS
)

def forget_rate(epoch):
    # Epochs 1–10 ramp from 0.02 to 0.20; then stay at 0.20.
    return MAX_FORGET_RATE * min((epoch + 1) / RAMPUP_EPOCHS, 1.0)

def train_epoch(net1, net2, loader, opt1, opt2, forget):
    net1.train()
    net2.train()
    loss_sum1 = 0.0
    loss_sum2 = 0.0
    correct1 = 0
    correct2 = 0
    total = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits1 = net1(x)
        logits2 = net2(x)

        # Per-example losses are required to select small-loss samples.
        losses1 = F.cross_entropy(logits1, y, reduction="none")
        losses2 = F.cross_entropy(logits2, y, reduction="none")

        remember = max(1, int((1.0 - forget) * len(y)))
        idx1 = torch.argsort(losses1)[:remember]
        idx2 = torch.argsort(losses2)[:remember]

        # Network 1 learns from the samples selected by network 2.
        loss1 = losses1[idx2].mean()
        # Network 2 learns from the samples selected by network 1.
        loss2 = losses2[idx1].mean()

        opt1.zero_grad(set_to_none=True)
        loss1.backward()
        opt1.step()

        opt2.zero_grad(set_to_none=True)
        loss2.backward()
        opt2.step()

        n = len(y)
        total += n
        loss_sum1 += losses1.detach().sum().item()
        loss_sum2 += losses2.detach().sum().item()
        correct1 += (logits1.argmax(1) == y).sum().item()
        correct2 += (logits2.argmax(1) == y).sum().item()

    return {
        "train_loss_net1": loss_sum1 / total,
        "train_loss_net2": loss_sum2 / total,
        "train_acc_net1": correct1 / total,
        "train_acc_net2": correct2 / total,
    }

@torch.no_grad()
def evaluate(net, loader):
    net.eval()
    loss_sum = 0.0
    correct = 0
    total = 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = net(x)
        loss_sum += F.cross_entropy(
            logits, y, reduction="sum"
        ).item()
        correct += (logits.argmax(1) == y).sum().item()
        total += len(y)
    return loss_sum / total, correct / total

# ---------------- TRAIN ----------------
history = []
best_val_acc = -1.0
best_epoch = None
best_net = None

for epoch in range(EPOCHS):
    fr = forget_rate(epoch)
    train_metrics = train_epoch(
        net1, net2, train_loader, opt1, opt2, fr
    )

    val_loss1, val_acc1 = evaluate(net1, val_loader)
    val_loss2, val_acc2 = evaluate(net2, val_loader)

    sched1.step()
    sched2.step()

    # Select the stronger network using SYNTHETIC validation accuracy.
    if val_acc1 >= val_acc2:
        selected_net = net1
        selected_name = "net1"
        selected_val_loss = val_loss1
        selected_val_acc = val_acc1
    else:
        selected_net = net2
        selected_name = "net2"
        selected_val_loss = val_loss2
        selected_val_acc = val_acc2

    row = {
        "epoch": epoch + 1,
        "forget_rate": fr,
        **train_metrics,
        "val_loss_net1": val_loss1,
        "val_acc_synthetic_net1": val_acc1,
        "val_loss_net2": val_loss2,
        "val_acc_synthetic_net2": val_acc2,
        "selected_network": selected_name,
        "selected_val_loss_synthetic": selected_val_loss,
        "selected_val_acc_synthetic": selected_val_acc,
    }
    history.append(row)

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"forget={fr:.3f} | "
        f"val1={val_acc1*100:.2f}% | "
        f"val2={val_acc2*100:.2f}% | "
        f"selected={selected_name}:{selected_val_acc*100:.2f}%"
    )

    if selected_val_acc > best_val_acc:
        best_val_acc = selected_val_acc
        best_epoch = epoch + 1
        best_net = selected_net
        torch.save({
            "epoch": best_epoch,
            "selected_network": selected_name,
            "val_acc_synthetic": best_val_acc,
            "model_state_dict": selected_net.state_dict(),
            "seed": SEED,
            "method": "coteaching",
        }, os.path.join(OUT_DIR, "t5_coteaching_best.pth"))

    # Save history every epoch so progress survives a session interruption.
    pd.DataFrame(history).to_csv(
        os.path.join(OUT_DIR, "t5_coteaching_history.csv"),
        index=False
    )

print(f"\nBest synthetic validation accuracy: {best_val_acc*100:.2f}%")
print("Best epoch:", best_epoch)

# ---------------- TEST PREDICTIONS ----------------
# Reload the saved best checkpoint rather than relying on the last epoch.
checkpoint = torch.load(
    os.path.join(OUT_DIR, "t5_coteaching_best.pth"),
    map_location=device
)
best_model = ResNet18(num_classes=10).to(device)
best_model.load_state_dict(checkpoint["model_state_dict"])
best_model.eval()

predictions = []
ids = []

@torch.no_grad()
def make_predictions():
    for x, batch_ids in test_loader:
        x = x.to(device, non_blocking=True)
        pred = best_model(x).argmax(dim=1).cpu().numpy()
        predictions.extend(pred.tolist())
        ids.extend(batch_ids.numpy().astype(int).tolist())

make_predictions()

submission = pd.DataFrame({"id": ids, "pred": predictions})
submission = submission.sort_values("id").reset_index(drop=True)

assert len(submission) == 10000, f"Expected 10000 rows, got {len(submission)}"
assert submission["id"].is_unique
assert submission["id"].min() == 0 and submission["id"].max() == 9999
assert submission["pred"].between(0, 9).all()

pred_path = os.path.join(OUT_DIR, "t5_best_synth.csv")
submission.to_csv(pred_path, index=False)

print("\nSaved files:")
print(os.path.join(OUT_DIR, "t5_coteaching_best.pth"))
print(os.path.join(OUT_DIR, "t5_coteaching_history.csv"))
print(pred_path)
print(submission.head())


Device: cuda
Dataset: /kaggle/input/datasets/tvjpro2/inter-iit-cv-ps2
Epoch 01/60 | forget=0.020 | val1=14.00% | val2=20.20% | selected=net2:20.20%
Epoch 02/60 | forget=0.040 | val1=22.22% | val2=21.76% | selected=net1:22.22%
Epoch 03/60 | forget=0.060 | val1=24.26% | val2=22.18% | selected=net1:24.26%
Epoch 04/60 | forget=0.080 | val1=27.44% | val2=18.90% | selected=net1:27.44%
Epoch 05/60 | forget=0.100 | val1=25.42% | val2=27.42% | selected=net2:27.42%
Epoch 06/60 | forget=0.120 | val1=26.40% | val2=31.14% | selected=net2:31.14%
Epoch 07/60 | forget=0.140 | val1=32.28% | val2=30.38% | selected=net1:32.28%
Epoch 08/60 | forget=0.160 | val1=34.88% | val2=30.98% | selected=net1:34.88%
Epoch 09/60 | forget=0.180 | val1=36.22% | val2=36.24% | selected=net2:36.24%
Epoch 10/60 | forget=0.200 | val1=32.22% | val2=35.22% | selected=net2:35.22%
Epoch 11/60 | forget=0.200 | val1=37.76% | val2=37.06% | selected=net1:37.76%
Epoch 12/60 | forget=0.200 | val1=38.48% | val2=37.80% | selected=net1:3

FileNotFoundError: Could not find t4_coteaching_best.pth under /kaggle/working or /kaggle/input